# Task 5 — Spark metadata stream to MongoDB

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | `screenshots/stage2_manifest.json` |
| Commands | `bash scripts/capture_spark_evidence.sh`; `bash scripts/capture_store_evidence.sh` |
| Stream | Spark consumes only `cpg.metadata`. |
| Storage contract | MongoDB upserts by `file_id` with checkpointed Kafka offsets. |
| Raw query output | [`metadata_evidence.txt`](../screenshots/mongodb/metadata_evidence.txt) |
```

## Approach and reasoning

Commands: `bash scripts/capture_spark_evidence.sh` and `bash scripts/capture_store_evidence.sh`. Spark owns the metadata path only: it reads `cpg.metadata`, persists checkpoint state, and writes one MongoDB document per `file_id`. That separation keeps metadata processing independent from the Neo4j graph connector.

## What this chapter proves

| Requirement | Evidence in this chapter |
|---|---|
| Spark metadata stream | The executed output records Spark offset and metadata document counts. |
| MongoDB upsert behavior | The chapter states the `file_id` replacement key and shows the final document. |
| UI-backed final state | The Mongo Express screenshot shows the replay final state after replacement. |


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/stage2_manifest.json').exists())
manifest = json.loads((ROOT / 'screenshots/stage2_manifest.json').read_text())
mongo = manifest['metrics']['mongodb']; spark = manifest['metrics']['spark']
print('MongoDB documents:', mongo['documents'])
print('unique file IDs/paths:', mongo['unique_file_ids'], '/', mongo['unique_file_paths'])
print('duplicate file_id groups:', mongo['duplicate_file_id_groups'])
print('Spark metadata offset:', spark['metadata_offset'])
print('completed micro-batch:', spark['completed_batch'])
assert mongo['documents'] == spark['metadata_offset'] == 5


MongoDB documents: 5
unique file IDs/paths: 5 / 5
duplicate file_id groups: 0
Spark metadata offset: 5
completed micro-batch: 0


## Final database UI evidence

```{admonition} Database UI evidence
:class: important

| Field | Value |
|---|---|
| Tool | Mongo Express |
| Phase represented | Canonical Stage 3 replay final state after metadata replacement. |
| UI action/query | Filter the target `file_id`. |
| Run date | `2026-07-17` |
| Result | `pass` |
| Backing artifacts | `screenshots/replay/mongodb_after_replay.json`; strict replay manifest. |
```

![Mongo Express final document after Stage 3 replay](../screenshots/replay/mongodb_after_replay.png)


## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Spark committed offset 5 and MongoDB contained five unique metadata documents with the exact repository and dataset commit. |
| Failed | That clean run had MongoDB query artifacts but no dedicated Mongo Express capture. |
| Resolution | The canonical Stage 3 replay captured a real, hash-validated Mongo Express final-state image after replacement; it is presented above with its actual phase. Replay/idempotence remains Task 6, not a Stage 2 claim. |
```
